Import all the necessary library. Read the patient diabetic record file. Read the header of the file.

In [1]:
import pandas as pd
import numpy as np


In [2]:

logistic_record=pd.read_csv(r"diabetes_record_logistic.csv")
print(logistic_record.head())

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


Replace the "zero" values of the column like age, glucose etc that is not possible in actual life witn NaN so the numpy recognize them and then replace them with the median of their own column 

In [3]:
zero_columns=["Glucose","BloodPressure","SkinThickness","BMI","Age"]
logistic_record[zero_columns]=logistic_record[zero_columns].replace(0,np.nan)
# logistic_record=logistic_record.dropna()
for column in zero_columns:
    logistic_record[column] = (
        logistic_record[column]
        .fillna(logistic_record[column].median())
    )
logistic_record.isnull().sum()



Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Set the value of the features (logistic_X) and the output(logistic_Y)

In [4]:
logistic_X=logistic_record.drop(["Outcome"],axis=1)
logistic_y=logistic_record["Outcome"]



Feature Scaling

In [5]:
logistic_mean=logistic_X.mean(axis=0)
logistic_std=logistic_X.std(axis=0)
normalization_l=(logistic_X-logistic_mean)/(logistic_std)
print(normalization_l)

     Pregnancies   Glucose  BloodPressure  SkinThickness   Insulin       BMI  \
0       0.639530  0.865481      -0.031969       0.670206 -0.692439  0.166511   
1      -0.844335 -1.204281      -0.527975      -0.012293 -0.692439 -0.851645   
2       1.233077  2.015348      -0.693310      -0.012293 -0.692439 -1.331632   
3      -0.844335 -1.072868      -0.527975      -0.694792  0.123221 -0.633469   
4      -1.141108  0.504094      -2.677331       0.670206  0.765337  1.548294   
..           ...       ...            ...            ...       ...       ...   
763     1.826623 -0.678627       0.298701       2.148954  0.869464  0.064695   
764    -0.547562  0.011293      -0.197304      -0.239793 -0.692439  0.631953   
765     0.342757 -0.021560      -0.031969      -0.694792  0.279412 -0.909825   
766    -0.844335  0.142707      -1.023980      -0.012293 -0.692439 -0.342567   
767    -0.844335 -0.941454      -0.197304       0.215206 -0.692439 -0.298932   

     DiabetesPedigreeFunction       Age

Sigmoid Function

In [6]:
def sigmoid(z):
    z= np.clip(z, -500, 500)
    return 1/(1+np.exp(-z))

Compute the Cost

In [7]:
def calculate_cost(X, y, w, b):

    m = X.shape[0]

    z = np.dot(X, w) + b
    prediction = sigmoid(z)

    eps = 1e-15
    prediction = np.clip(prediction, eps, 1-eps)

    cost = (
        -y * np.log(prediction)
        -(1-y) * np.log(1-prediction)
    )

    return np.sum(cost) / m

Compute Gradient

In [8]:
def compute_gradient_l(x,y,w,b,sigmoid_fun):
    m=x.shape[0]
    z=(np.dot(x,w)+b)
    prediction=sigmoid_fun(z)
    subtraction=prediction-y
    w=np.dot(x.T,subtraction)/m
    b=np.sum(subtraction)/m
    return w,b

Compute Graient Descent

In [9]:
def gradient_descent_l(x,y,w,b,alpha,gradient_fun,sigmoid_fun,num_iter_l,cost_function):
    for i in range(num_iter_l):
        compute_cost=cost_function(x,y,w,b)
        dj_dw, dj_db = gradient_fun(x, y, w, b,sigmoid_fun)   
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db 
        if i % 1000 == 0:
                    print(f"Iteration {i}: Cost={compute_cost:.6f}")
    print("-------------------------\n  --------------------")                
    return w,b

Prediction compare the actual and the target values

In [10]:
num_iter_l=10000
logistic_X = normalization_l.values
logistic_y=logistic_y.values
alpha_l=0.9
logistic_b=0
m=logistic_X.shape[0]
logistic_w=np.array([0,0,0,0,0,0,0,0])
w,b=gradient_descent_l(logistic_X,logistic_y,logistic_w,logistic_b,alpha_l,compute_gradient_l,sigmoid,num_iter_l,calculate_cost) 
prediction=(np.dot(logistic_X,w)+b) 
sig_prediction=sigmoid(prediction)
predictions = (sig_prediction >= 0.5).astype(int)

logistic_y = logistic_y.flatten()

for i in range(m):
    print(f"Patient {i+1} -> Probability: {sig_prediction[i]:.2f} | Predicted Class: {predictions[i]} | Actual Target: {logistic_y[i]}")



Iteration 0: Cost=0.693147
Iteration 1000: Cost=0.462487
Iteration 2000: Cost=0.462487
Iteration 3000: Cost=0.462487
Iteration 4000: Cost=0.462487
Iteration 5000: Cost=0.462487
Iteration 6000: Cost=0.462487
Iteration 7000: Cost=0.462487
Iteration 8000: Cost=0.462487
Iteration 9000: Cost=0.462487
-------------------------
  --------------------
Patient 1 -> Probability: 0.74 | Predicted Class: 1 | Actual Target: 1
Patient 2 -> Probability: 0.04 | Predicted Class: 0 | Actual Target: 0
Patient 3 -> Probability: 0.82 | Predicted Class: 1 | Actual Target: 1
Patient 4 -> Probability: 0.04 | Predicted Class: 0 | Actual Target: 0
Patient 5 -> Probability: 0.90 | Predicted Class: 1 | Actual Target: 1
Patient 6 -> Probability: 0.14 | Predicted Class: 0 | Actual Target: 0
Patient 7 -> Probability: 0.05 | Predicted Class: 0 | Actual Target: 1
Patient 8 -> Probability: 0.42 | Predicted Class: 0 | Actual Target: 0
Patient 9 -> Probability: 0.72 | Predicted Class: 1 | Actual Target: 1
Patient 10 -> P

Find accuracy of the model

In [11]:
# 6. FIX: Use 'predictions' (integers) to evaluate true model accuracy
accuracy = np.mean(predictions == logistic_y) * 100
print(f"\nFinal Model Accuracy: {accuracy:.2f}%")


Final Model Accuracy: 77.47%


User Input. user enter the neceessary information and model predict whether the patient has the diabetes or not.

In [12]:

print("        Diabetes Prediction System        ")

Pregnancies = float(input("Pregnancies: "))
Glucose = float(input("Glucose: "))
BloodPressure = float(input("Blood Pressure: "))
SkinThickness = float(input("Skin Thickness: "))
Insulin = float(input("Insulin: "))
BMI = float(input("BMI: "))

DiabetesPedigreeFunction = float(
    input("Diabetes Pedigree Function: ")
)

Age = float(input("Age: "))

new_patient = pd.DataFrame({
    "Pregnancies": [Pregnancies],
    "Glucose": [Glucose],
    "BloodPressure": [BloodPressure],
    "SkinThickness": [SkinThickness],
    "Insulin": [Insulin],
    "BMI": [BMI],
    "DiabetesPedigreeFunction": [
        DiabetesPedigreeFunction
    ],
    "Age": [Age]
})

# Normalize using training mean/std
new_patient_normalized = (
    new_patient - logistic_mean
) / logistic_std

# Convert to NumPy
new_patient_record = (
    new_patient_normalized.values.flatten()
)

# Calculate probability
probability = sigmoid(
    np.dot(w, new_patient_record) + b
)

# Apply threshold
prediction = int(probability >= 0.5)

print("Prediction Results")

print(
    f"Diabetes Probability: "
    f"{probability * 100:.2f}%"
)

if prediction == 1:
    print("Prediction: Diabetes")
else:
    print("Prediction: No Diabetes")

        Diabetes Prediction System        
Prediction Results
Diabetes Probability: 73.34%
Prediction: Diabetes
